 Derived from
 
 https://www.axonlab.org/hcph-sops/data-management/edf-to-bids/
 
 https://github.com/TheAxonLab/hcph-sops/blob/mkdocs/code/eyetracking/convert.py

Make sure the python version >=3.7 to support the statement

In [4]:
from __future__ import annotations 
from pathlib import Path
import pandas as pd
import numpy as np
from pyedfread import read_edf
from collections import defaultdict
from itertools import product, groupby
from warnings import warn
import re

In [5]:
# Global variable from 
# https://github.com/TheAxonLab/hcph-sops/blob/mkdocs/code/eyetracking/convert.py

# If setting WRITE_RAW_EDF as True, no preprocessing will be conducted on the edf data
WRITE_RAW_EDF = True
# -------------------------------------------------------------------------------
DEFAULT_EYE = "right"
DEFAULT_FREQUENCY = 1000 #It is 1000
DEFAULT_MODE = "P-CR"
DEFAULT_SCREEN = (0, 800, 0, 600)

# EyeLink calibration coordinates from
# https://www.sr-research.com/calibration-coordinate-calculator/
# Affect the performance?
EYELINK_CALIBRATION_COORDINATES = [
    (400, 300),
    (400, 51),
    (400, 549),
    (48, 300),
    (752, 300),
    (48, 51),
    (752, 51),
    (48, 549),
    (752, 549),
    (224, 176),
    (576, 176),
    (224, 424),
    (576, 424),
]

EYE_CODE_MAP = defaultdict(lambda: "unknown", {"R": "right", "L": "left", "RL": "both"})
EDF2BIDS_COLUMNS = {
    "g": '',
    "p": "pupil",
    "h": "href",
    "r": "raw",
    "fg": "fast",
    "fh": "fast_href",
    "fr": "fast_raw",
}

BIDS_COLUMNS_ORDER = (
    [f"eye{num}_{c}_coordinate" for num, c in product((1, 2), ("x", "y"))]
    + [f"eye{num}_pupil_size" for num in (1, 2)]
    + [f"eye{num}_pupil_{c}_coordinate" for num, c in product((1, 2), ("x", "y"))]
    + [f"eye{num}_fixation" for num in (1, 2)]
    + [f"eye{num}_saccade" for num in (1, 2)]
    + [f"eye{num}_blink" for num in (1, 2)]
    + [f"eye{num}_href_{c}_coordinate" for num, c in product((1, 2), ("x", "y"))]
    + [f"eye{num}_{c}_velocity" for num, c in product((1, 2), ("x", "y"))]
    + [f"eye{num}_href_{c}_velocity" for num, c in product((1, 2), ("x", "y"))]
    + [f"eye{num}_raw_{c}_velocity" for num, c in product((1, 2), ("x", "y"))]
    + [f"fast_{c}_velocity" for c in ("x", "y")]
    + [f"fast_{kind}_{c}_velocity" for kind, c in product(("href", "raw"), ("x", "y"))]
    + [f"screen_ppdeg_{c}_coordinate" for c in ("x", "y")]
    + ["timestamp"]
)

Read in the edf file

In [6]:
subject_idx = 2
T_idx = 1

DATA_PATH = Path(f"/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-{subject_idx:03d}/et")

# Find the .edf file automatically
edf_files = list(DATA_PATH.glob(f"*T{T_idx}*.EDF"))
if not edf_files:
    edf_files = list(DATA_PATH.glob("*.EDF"))
    
if edf_files:
    edf_name = edf_files[0].name
    print(f"Found EDF file: {edf_name}")
else:
    raise FileNotFoundError(f"No EDF file found in {DATA_PATH}")

file_path = str(DATA_PATH / edf_name)
print(file_path)
ori_recording, ori_events, ori_messages = read_edf(file_path)
# The first timestamp of  `recording`
print(f" {ori_recording[100000:100100]}")
# print(ori_messages)
# print(ori_events)
# print(messages)
ori_messages = ori_messages.rename(
    columns={
        # Normalize weird header names generated by pyedfread
        "message": "trialid",
        "trial": "trial",
        # Convert some BIDS columns
        "time": "timestamp",
    }
)

recording = ori_recording
messages = ori_messages
events = ori_events
print(f'\nThe entire info of `message`: \n{messages[20:80]}')
recording.columns

Found EDF file: 002_mreyetrack_4points_2025-12-19_17h34.06.389.EDF
/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-002/et/002_mreyetrack_4points_2025-12-19_17h34.06.389.EDF
loadEvents = 1
              time  px_left  px_right  py_left  py_right  hx_left  hx_right  \
100000  5415568.0 -32768.0   -1040.0 -32768.0   -6202.0 -32768.0     -81.0   
100001  5415569.0 -32768.0   -1034.0 -32768.0   -6210.0 -32768.0     -77.0   
100002  5415570.0 -32768.0   -1026.0 -32768.0   -6210.0 -32768.0     -71.0   
100003  5415571.0 -32768.0   -1018.0 -32768.0   -6204.0 -32768.0     -64.0   
100004  5415572.0 -32768.0   -1022.0 -32768.0   -6196.0 -32768.0     -67.0   
...           ...      ...       ...      ...       ...      ...       ...   
100095  5415663.0 -32768.0   -1234.0 -32768.0   -6249.0 -32768.0    -240.0   
100096  5415664.0 -32768.0   -1238.0 -32768.0   -6248.0 -32768.0    -243.0   
100097  5415665.0 -32768.0   -1240.0 -32768.0   -6248.0 -32768.0    -244.0   
100098  5415666.0 -32768.0   

Index(['time', 'px_left', 'px_right', 'py_left', 'py_right', 'hx_left',
       'hx_right', 'hy_left', 'hy_right', 'pa_left', 'pa_right', 'gx_left',
       'gx_right', 'gy_left', 'gy_right', 'rx', 'ry', 'gxvel_left',
       'gxvel_right', 'gyvel_left', 'gyvel_right', 'hxvel_left', 'hxvel_right',
       'hyvel_left', 'hyvel_right', 'rxvel_left', 'rxvel_right', 'ryvel_left',
       'ryvel_right', 'fgxvel', 'fgyvel', 'fhxvel', 'fhyvel', 'frxvel',
       'fryvel', 'flags', 'input', 'buttons', 'htype', 'errors'],
      dtype='object')

# 1 Parsing the messages

In [7]:
messages = messages.rename(
    columns={c: c.strip() for c in messages.columns.values}
).drop_duplicates()

In [8]:
# Extract calibration headers
_cal_hdr = ori_messages.trialid.str.startswith("!CAL")
calibration = ori_messages[_cal_hdr]
# messages = messages.drop(messages.index[_cal_hdr])
print(calibration)

    timestamp  trial                                            trialid
1     4752963     -1  !CAL \n>>>>>>> CALIBRATION (HV5,P-CR) FOR RIGH...
2     4752963     -1                           !CAL Calibration points:
3     4752963     -1                 !CAL -8.1, -40.9         0,      0
4     4752963     -1                 !CAL -6.9, -55.7         0,  -1726
5     4752963     -1                 !CAL -8.5, -24.7         0,   1726
6     4752963     -1                !CAL -32.0, -37.7     -2439,      0
7     4752963     -1                !CAL  17.0, -39.1      2439,      0
8     4752963     -1  !CAL eye check box: (L,R,T,B)\n\t  -37    22  ...
9     4752963     -1  !CAL href cal range: (L,R,T,B)\n\t-3659  3659 ...
10    4752963     -1  !CAL Cal coeff:(X=a+bx+cy+dxx+eyy,Y=f+gx+goaly...
11    4752963     -1    !CAL Prenormalize: offx, offy = -8.0857 -40.878
12    4752963     -1         !CAL Gains: cx:97.645 lx:106.941 rx:96.066
13    4752963     -1         !CAL Gains: cy:84.279 ty:118.628 by

In [9]:
# Extracting the StartTime and StopTime metadata.
message_first_trigger = '!MODE RECORD CR 1000 2 0 R'
message_last_trigger = 'ET: eye-tracker stopped'
metadata = {
    'StopTime': None,
    'StartTime': None
}

# Find Start time
start_rows = messages.trialid.str.contains(
    message_first_trigger, case=False, regex=True
)
stop_rows = messages.trialid.str.contains(
    message_last_trigger, case=False, regex=True
)


# Extract calibration headers
_cal_hdr = messages.trialid.str.startswith("!CAL")
calibration = messages[_cal_hdr]
messages = messages.drop(messages.index[_cal_hdr])

# Pick the LAST of the start messages
metadata["StartTime"] = (
    int(messages[start_rows].timestamp.values[-1])
    if start_rows.any()
    else None
)

# Pick the FIRST of the stop messages
metadata["StopTime"] = (
    int(messages[stop_rows].timestamp.values[0])
    if stop_rows.any()
    else None
)

# Drop start and stop messages from messages dataframe
messages = messages.loc[~start_rows & ~stop_rows, :]

metadata

/tmp/ipykernel_4104226/906092332.py:25: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  int(messages[start_rows].timestamp.values[-1])


{'StopTime': None, 'StartTime': 5315568}

In [10]:
# Extracting basic metadata.
# !MODE RECORD CR 1000 2 0 R

mode_record = messages.trialid.str.startswith("!MODE RECORD")

meta_record = {
    "freq": DEFAULT_FREQUENCY,
    "mode": DEFAULT_MODE,
    "eye": DEFAULT_EYE,
}

if mode_record.any():
    try:
        meta_record = re.match(
            r"\!MODE RECORD (?P<mode>\w+) (?P<freq>\d+) \d \d (?P<eye>[RL]+)",
            messages[mode_record].trialid.iloc[-1].strip(),
        ).groupdict()

        meta_record["eye"] = EYE_CODE_MAP[meta_record["eye"]]
        meta_record["mode"] = (
            "P-CR" if meta_record["mode"] == "CR" else meta_record["mode"]
        )
    except AttributeError:
        warn(
            "Error extracting !MODE RECORD message, "
            "using default frequency, mode, and eye"
        )
    finally:
        messages = messages.loc[~mode_record]

eye = (
    ("right", "left") if meta_record["eye"] == "both" else (meta_record["eye"],)
)

metadata["SamplingFrequency"] = int(meta_record["freq"])
metadata["EyeTrackingMethod"] = meta_record["mode"]
metadata["RecordedEye"] = meta_record["eye"]

In [11]:
# Extracting screen parameters.
# GAZE_COORDS 0.00 0.00 800.00 600.00

# Extract GAZE_COORDS message signaling start of recording
gaze_msg = messages.trialid.str.startswith("GAZE_COORDS")

metadata["ScreenAOIDefinition"] = [
    "square",
    DEFAULT_SCREEN,
]
if gaze_msg.any():
    try:
        gaze_record = re.match(
            r"GAZE_COORDS (\d+\.\d+) (\d+\.\d+) (\d+\.\d+) (\d+\.\d+)",
            messages[gaze_msg].trialid.iloc[-1].strip(),
        ).groups()
        metadata["ScreenAOIDefinition"][1] = [
            int(round(float(gaze_record[0]))),
            int(round(float(gaze_record[2]))),
            int(round(float(gaze_record[1]))),
            int(round(float(gaze_record[3]))),
        ]
    except AttributeError:
        warn("Error extracting GAZE_COORDS")
    finally:
        messages = messages.loc[~gaze_msg]
        
print(metadata)

{'StopTime': None, 'StartTime': 5315568, 'SamplingFrequency': 1000, 'EyeTrackingMethod': 'P-CR', 'RecordedEye': 'right', 'ScreenAOIDefinition': ['square', [0, 800, 0, 600]]}


In [12]:
# Extracting parameters of the pupil fit model.
# ELCL_PROC ELLIPSE (5)
# ELCL_EFIT_PARAMS 1.01 4.00  0.15 0.05  0.65 0.65  0.00 0.00 0.30
# Extract ELCL_PROC AND ELCL_EFIT_PARAMS to extract pupil fit method
pupilfit_msg = messages.trialid.str.startswith("ELCL_PROC")

if pupilfit_msg.any():
    try:
        pupilfit_method = [
            val
            for val in messages[pupilfit_msg]
            .trialid.iloc[-1]
            .strip()
            .split(" ")[1:]
            if val
        ]
        metadata["PupilFitMethod"] = pupilfit_method[0].lower()
        metadata["PupilFitMethodNumberOfParameters"] = int(
            pupilfit_method[1].strip("(").strip(")")
        )
    except AttributeError:
        warn("Error extracting ELCL_PROC (pupil fitting method)")
    finally:
        messages = messages.loc[~pupilfit_msg]

pupilfit_msg_params = messages.trialid.str.startswith("ELCL_EFIT_PARAMS")
if pupilfit_msg_params.any():
    rows = messages[pupilfit_msg_params]
    row = rows.trialid.values[-1].strip().split(" ")[1:]
    try:
        metadata["PupilFitParameters"] = [
            tuple(float(val) for val in vals)
            for k, vals in groupby(row, key=bool)
            if k
        ]
    except AttributeError:
        warn("Error extracting ELCL_EFIT_PARAMS (pupil fitting parameters)")
    finally:
        messages = messages.loc[~pupilfit_msg_params]
        
metadata

{'StopTime': None,
 'StartTime': 5315568,
 'SamplingFrequency': 1000,
 'EyeTrackingMethod': 'P-CR',
 'RecordedEye': 'right',
 'ScreenAOIDefinition': ['square', [0, 800, 0, 600]],
 'PupilFitMethod': 'ellipse',
 'PupilFitMethodNumberOfParameters': 5,
 'PupilFitParameters': [(1.01, 4.0),
  (0.15, 0.05),
  (0.65, 0.65),
  (0.0, 0.0, 0.3)]}

In [13]:
# Calibration validation.
# VALIDATE R 4POINT 4 RIGHT at 752,300 OFFSET 0.35 deg. -8.7,-3.8 pix.
# Extract VALIDATE messages for a calibration validation
validation_msg = messages.trialid.str.startswith("VALIDATE")

if validation_msg.any():
    metadata["ValidationPosition"] = []
    metadata["ValidationErrors"] = []

for i_row, validate_row in enumerate(messages[validation_msg].trialid.values):
    prefix, suffix = validate_row.split("OFFSET")
    validation_eye = (
        f"eye{eye.index('right') + 1}"
        if "RIGHT" in prefix
        else f"eye{eye.index('left') + 1}"
    )
    validation_coords = [
        int(val.strip())
        for val in prefix.rsplit("at", 1)[-1].split(",")
        if val.strip()
    ]
    metadata["ValidationPosition"].append(
        [validation_eye, validation_coords]
    )

    validate_values = [
        float(val)
        for val in re.match(
            r"(-?\d+\.\d+) deg\.\s+(-?\d+\.\d+),(-?\d+\.\d+) pix\.",
            suffix.strip(),
        ).groups()
    ]

    metadata["ValidationErrors"].append(
        (validation_eye, validate_values[0], tuple(validate_values[1:]))
    )
messages = messages.loc[~validation_msg]

print(messages)
print(metadata)

     timestamp  trial                                            trialid
0      4702585     -1                        ET: Start experiment 'dots'
34     4886972     -1  NO Reply is disabled for function eyelink_cal_...
35     5315561     -1                                ET: start recording
36     5315567     -1                               RECCFG CR 1000 2 0 R
37     5315567     -1                                      ELCLCFG TOWER
..         ...    ...                                                ...
237    5967472     -1                            ET: Start routine 'dot'
238    5972470     -1                            ET: Start routine 'dot'
240    5972471     -1                            ET: Start routine 'dot'
242    5977468     -1                 ET: Prepare to start routine 'end'
243    5989502     -1                            ET: eye tracker stopped

[185 rows x 3 columns]
{'StopTime': None, 'StartTime': 5315568, 'SamplingFrequency': 1000, 'EyeTrackingMethod': 'P-CR', 'Re

In [14]:
# Extracting final bits of metadata.
# Extract THRESHOLDS messages prior recording and process last
thresholds_msg = messages.trialid.str.startswith("THRESHOLDS")
if thresholds_msg.any():
    metadata["PupilThreshold"] = [None] * len(eye)
    metadata["CornealReflectionThreshold"] = [None] * len(eye)
    thresholds_chunks = (
        messages[thresholds_msg].trialid.iloc[-1].strip().split(" ")[1:]
    )
    eye_index = eye.index(EYE_CODE_MAP[thresholds_chunks[0]])
    metadata["PupilThreshold"][eye_index] = int(thresholds_chunks[-2])
    metadata["CornealReflectionThreshold"][eye_index] = int(
        thresholds_chunks[-1]
    )
messages = messages.loc[~thresholds_msg]
print(messages)
print(metadata)

     timestamp  trial                                            trialid
0      4702585     -1                        ET: Start experiment 'dots'
34     4886972     -1  NO Reply is disabled for function eyelink_cal_...
35     5315561     -1                                ET: start recording
36     5315567     -1                               RECCFG CR 1000 2 0 R
37     5315567     -1                                      ELCLCFG TOWER
..         ...    ...                                                ...
237    5967472     -1                            ET: Start routine 'dot'
238    5972470     -1                            ET: Start routine 'dot'
240    5972471     -1                            ET: Start routine 'dot'
242    5977468     -1                 ET: Prepare to start routine 'end'
243    5989502     -1                            ET: eye tracker stopped

[184 rows x 3 columns]
{'StopTime': None, 'StartTime': 5315568, 'SamplingFrequency': 1000, 'EyeTrackingMethod': 'P-CR', 'Re

In [15]:
# Flush the remaining messages as a metadata entry.
# Consume the remainder of messages

if not messages.empty:
    metadata["LoggedMessages"] = [
        (int(msg_timestamp), msg.strip())
        for msg_timestamp, msg in messages[["timestamp", "trialid"]].values
    ]
    
print(messages)
print(metadata)

     timestamp  trial                                            trialid
0      4702585     -1                        ET: Start experiment 'dots'
34     4886972     -1  NO Reply is disabled for function eyelink_cal_...
35     5315561     -1                                ET: start recording
36     5315567     -1                               RECCFG CR 1000 2 0 R
37     5315567     -1                                      ELCLCFG TOWER
..         ...    ...                                                ...
237    5967472     -1                            ET: Start routine 'dot'
238    5972470     -1                            ET: Start routine 'dot'
240    5972471     -1                            ET: Start routine 'dot'
242    5977468     -1                 ET: Prepare to start routine 'end'
243    5989502     -1                            ET: eye tracker stopped

[184 rows x 3 columns]
{'StopTime': None, 'StartTime': 5315568, 'SamplingFrequency': 1000, 'EyeTrackingMethod': 'P-CR', 'Re

# 2 Parsing the recording dataframe

In [16]:
recording = ori_recording
ori_recording

,time,px_left,px_right,py_left,py_right,hx_left,hx_right,hy_left,hy_right,pa_left,...,fgyvel,fhxvel,fhyvel,frxvel,fryvel,flags,input,buttons,htype,errors
0,5315568.0,-32768.0,362.0,-32768.0,-3864.0,-32768.0,1115.0,-32768.0,1068.0,-32768.0,...,0.0,4.162697e-41,0.0,4.591354e-41,4.162697e-41,32641.0,32768.0,0.0,-32768.0,0.0
1,5315569.0,-32768.0,359.0,-32768.0,-3864.0,-32768.0,1113.0,-32768.0,1068.0,-32768.0,...,0.0,4.162697e-41,0.0,4.591354e-41,4.162697e-41,24449.0,32768.0,0.0,-32768.0,0.0
2,5315570.0,-32768.0,355.0,-32768.0,-3869.0,-32768.0,1109.0,-32768.0,1064.0,-32768.0,...,0.0,4.162697e-41,0.0,4.591354e-41,4.162697e-41,24449.0,32768.0,0.0,-32768.0,0.0
3,5315571.0,-32768.0,351.0,-32768.0,-3875.0,-32768.0,1107.0,-32768.0,1060.0,-32768.0,...,0.0,4.162697e-41,0.0,4.591354e-41,4.162697e-41,24449.0,32768.0,0.0,-32768.0,0.0
4,5315572.0,-32768.0,349.0,-32768.0,-3880.0,-32768.0,1105.0,-32768.0,1055.0,-32768.0,...,0.0,4.162697e-41,0.0,4.591354e-41,4.162697e-41,24449.0,32768.0,0.0,-32768.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
662897,5978465.0,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,-7936.0,-32768.0,-7936.0,-32768.0,...,NaN,0.000000e+00,0.0,4.591354e-41,4.162697e-41,24449.0,32768.0,0.0,-32768.0,0.0
662898,5978466.0,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,-7936.0,-32768.0,-7936.0,-32768.0,...,NaN,0.000000e+00,0.0,4.591354e-41,4.162697e-41,24449.0,32768.0,0.0,-32768.0,0.0
662899,5978467.0,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,-7936.0,-32768.0,-7936.0,-32768.0,...,NaN,0.000000e+00,0.0,4.591354e-41,4.162697e-41,24449.0,32768.0,0.0,-32768.0,0.0
662900,5978468.0,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,-7936.0,-32768.0,-7936.0,-32768.0,...,NaN,0.000000e+00,0.0,4.591354e-41,4.162697e-41,32641.0,32768.0,0.0,-32768.0,0.0


In [17]:
# Curation of the input dataframe
# Normalize timestamps (should be int and strictly positive)
recording = recording.astype({"time": int})
recording = recording[recording["time"] > 0]
raw_recording_len = len(recording)
print(f'raw_recording length: {raw_recording_len}')

recording = recording.rename(
    columns={
#         # Fix buggy header names generated by pyedfread
#         "fhxyvel": "fhxvel",
#         "frxyvel": "frxvel",
        # Normalize weird header names generated by pyedfread
        "rx": "screen_ppdeg_x_coordinate",
        "ry": "screen_ppdeg_y_coordinate",
        # Convert some BIDS columns
        "time": "timestamp",
    }
)

# Split extra columns from the dataframe
extra = recording[["flags", "input", "htype"]]
recording = recording.drop(columns=["flags", "input", "htype"])
print(len(recording))

# Remove columns that are always very close to zero
recording = recording.loc[:, (recording.abs() > 1e-8).any(axis=0)]
# Remove columns that are always 1e8 or more
recording = recording.loc[:, (recording.abs() < 1e8).any(axis=0)]
# Replace unreasonably high values with NaNs
recording = recording.replace({1e8: np.nan})

assert len(recording) == raw_recording_len

raw_recording length: 662902
662902


In [18]:
# Remove columns that do not apply (e.g., only one eye recorded).
# Drop one eye's columns if not interested in "both"
print(f'The eye we take care of {eye}')
remove_eye = set(("left", "right")) - set(eye)
if remove_eye:
    remove_eye = remove_eye.pop()  # Drop set decoration
    recording = recording.reindex(
        columns=[c for c in recording.columns if remove_eye not in c]
    )
    
columns = recording.columns
print("Columns:")
print(columns)
recording

The eye we take care of ('right',)
Columns:
Index(['timestamp', 'px_right', 'py_right', 'hx_right', 'hy_right', 'pa_right',
       'gx_right', 'gy_right', 'screen_ppdeg_x_coordinate',
       'screen_ppdeg_y_coordinate', 'gxvel_right', 'gyvel_right',
       'hxvel_right', 'hyvel_right', 'rxvel_right', 'ryvel_right', 'fgxvel'],
      dtype='object')


,timestamp,px_right,py_right,hx_right,hy_right,pa_right,gx_right,gy_right,screen_ppdeg_x_coordinate,screen_ppdeg_y_coordinate,gxvel_right,gyvel_right,hxvel_right,hyvel_right,rxvel_right,ryvel_right,fgxvel
0,5315568,362.0,-3864.0,1115.0,1068.0,1342.0,560.900024,454.100006,38.000000,38.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,5315569,359.0,-3864.0,1113.0,1068.0,1341.0,560.599976,454.200012,38.000000,38.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,5315570,355.0,-3869.0,1109.0,1064.0,1340.0,560.099976,453.600006,38.000000,38.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,5315571,351.0,-3875.0,1107.0,1060.0,1336.0,559.700012,452.899994,38.000000,38.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,5315572,349.0,-3880.0,1105.0,1055.0,1332.0,559.500000,452.299988,38.000000,38.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
662897,5978465,-32768.0,-32768.0,-7936.0,-7936.0,0.0,NaN,NaN,38.200001,38.700001,NaN,NaN,NaN,NaN,NaN,NaN,NaN
662898,5978466,-32768.0,-32768.0,-7936.0,-7936.0,0.0,NaN,NaN,38.200001,38.700001,NaN,NaN,NaN,NaN,NaN,NaN,NaN
662899,5978467,-32768.0,-32768.0,-7936.0,-7936.0,0.0,NaN,NaN,38.200001,38.700001,NaN,NaN,NaN,NaN,NaN,NaN,NaN
662900,5978468,-32768.0,-32768.0,-7936.0,-7936.0,0.0,NaN,NaN,38.200001,38.700001,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
# Clean-up pupil size and gaze position. 
# These are the parameters we most likely we care for, so special curation is applied:
screen_resolution = [800, 600]

for eyenum, eyename in enumerate(eye):
    # Clean-up implausible values for pupil area (pa)
    recording.loc[
        recording[f"pa_{eyename}"] < 1, f"pa_{eyename}"
    ] = np.nan
    recording = recording.rename(
        columns={f"pa_{eyename}": f"eye{eyenum + 1}_pupil_size"}
    )
    print(f"pa_{eyename} renamed as: eye{eyenum + 1}_pupil_size")
    # Clean-up implausible values for gaze x position
    recording.loc[
        (recording[f"gx_{eyename}"] < 0)
        | (recording[f"gx_{eyename}"] > screen_resolution[0]),
        f"gx_{eyename}",
    ] = np.nan
    # Clean-up implausible values for gaze y position
    recording.loc[
        (recording[f"gy_{eyename}"] <= 0)
        | (recording[f"gy_{eyename}"] > screen_resolution[1]),
        f"gy_{eyename}",
    ] = np.nan
    
print(recording)
assert len(recording) == raw_recording_len

pa_right renamed as: eye1_pupil_size
        timestamp  px_right  py_right  hx_right  hy_right  eye1_pupil_size  \
0         5315568     362.0   -3864.0    1115.0    1068.0           1342.0   
1         5315569     359.0   -3864.0    1113.0    1068.0           1341.0   
2         5315570     355.0   -3869.0    1109.0    1064.0           1340.0   
3         5315571     351.0   -3875.0    1107.0    1060.0           1336.0   
4         5315572     349.0   -3880.0    1105.0    1055.0           1332.0   
...           ...       ...       ...       ...       ...              ...   
662897    5978465  -32768.0  -32768.0   -7936.0   -7936.0              NaN   
662898    5978466  -32768.0  -32768.0   -7936.0   -7936.0              NaN   
662899    5978467  -32768.0  -32768.0   -7936.0   -7936.0              NaN   
662900    5978468  -32768.0  -32768.0   -7936.0   -7936.0              NaN   
662901    5978469  -32768.0  -32768.0   -7936.0   -7936.0              NaN   

          gx_right    gy_r

In [20]:
# Munging columns to comply with BIDS. 
# At this point, the dataframe is almost ready for writing out as BIDS.
# Interpolate BIDS column names
columns = list(
    set(recording.columns)
    - set(
        (
            "timestamp",
            "screen_ppdeg_x_coordinate",
            "screen_ppdeg_y_coordinate",
            "eye1_pupil_size",#pa
            "eye2_pupil_size",#pa
        )
    )
)
bids_columns = []
for eyenum, eyename in enumerate(eye):
    for name in columns:
        colprefix = f"eye{eyenum + 1}" if name.endswith(f"_{eyename}") else ""
        _newname = name.split("_")[0]
        _newname = re.sub(r"([xy])$", r"_\1_coordinate", _newname)
        _newname = re.sub(r"([xy])vel$", r"_\1_velocity", _newname)
        _newname = _newname.split("_", 1)
        _newname[0] = EDF2BIDS_COLUMNS[_newname[0]]
        _newname.insert(0, colprefix)
        bids_columns.append("_".join((_n for _n in _newname if _n)))

# Rename columns to be BIDS-compliant
recording = recording.rename(columns=dict(zip(columns, bids_columns)))

# Reorder columns to render nicely (tracking first, pupil size after)
columns = sorted(
    set(recording.columns.values).intersection(BIDS_COLUMNS_ORDER),
    key=lambda entry: BIDS_COLUMNS_ORDER.index(entry),
)
columns += [c for c in recording.columns.values if c not in columns]
recording = recording.reindex(columns=columns)

print(recording)
assert len(recording) == raw_recording_len

        eye1_x_coordinate  eye1_y_coordinate  eye1_pupil_size  \
0              560.900024         454.100006           1342.0   
1              560.599976         454.200012           1341.0   
2              560.099976         453.600006           1340.0   
3              559.700012         452.899994           1336.0   
4              559.500000         452.299988           1332.0   
...                   ...                ...              ...   
662897                NaN                NaN              NaN   
662898                NaN                NaN              NaN   
662899                NaN                NaN              NaN   
662900                NaN                NaN              NaN   
662901                NaN                NaN              NaN   

        eye1_pupil_x_coordinate  eye1_pupil_y_coordinate  \
0                         362.0                  -3864.0   
1                         359.0                  -3864.0   
2                         355.0        

# 3 Parsing the calibration messages

In [21]:
print(calibration)

    timestamp  trial                                            trialid
1     4752963     -1  !CAL \n>>>>>>> CALIBRATION (HV5,P-CR) FOR RIGH...
2     4752963     -1                           !CAL Calibration points:
3     4752963     -1                 !CAL -8.1, -40.9         0,      0
4     4752963     -1                 !CAL -6.9, -55.7         0,  -1726
5     4752963     -1                 !CAL -8.5, -24.7         0,   1726
6     4752963     -1                !CAL -32.0, -37.7     -2439,      0
7     4752963     -1                !CAL  17.0, -39.1      2439,      0
8     4752963     -1  !CAL eye check box: (L,R,T,B)\n\t  -37    22  ...
9     4752963     -1  !CAL href cal range: (L,R,T,B)\n\t-3659  3659 ...
10    4752963     -1  !CAL Cal coeff:(X=a+bx+cy+dxx+eyy,Y=f+gx+goaly...
11    4752963     -1    !CAL Prenormalize: offx, offy = -8.0857 -40.878
12    4752963     -1         !CAL Gains: cx:97.645 lx:106.941 rx:96.066
13    4752963     -1         !CAL Gains: cy:84.279 ty:118.628 by

In [22]:
# Parse calibration metadata
metadata["CalibrationCount"] = 0
if not calibration.empty:
    warn("Calibration of more than one eye is not implemented")
    calibration.trialid = calibration.trialid.str.replace("!CAL", "")
    calibration.trialid = calibration.trialid.str.strip()

    metadata["CalibrationLog"] = list(
        zip(
            calibration.timestamp.values.astype(int),
            calibration.trialid.values,
        )
    )

    calibrations_msg = calibration.trialid.str.startswith(
        "VALIDATION"
    ) & calibration.trialid.str.contains("ERROR")
    metadata["CalibrationCount"] = calibrations_msg.sum()

    calibration_last = calibration.index[calibrations_msg][-1]
    try:
        meta_calib = re.match(
            r"VALIDATION (?P<ctype>[\w\d]+) (?P<eyeid>[RL]+) (?P<eye>RIGHT|LEFT) "
            r"(?P<result>\w+) ERROR (?P<avg>-?\d+\.\d+) avg\. (?P<max>-?\d+\.\d+) max\s+"
            r"OFFSET (?P<offsetdeg>-?\d+\.\d+) deg\. "
            r"(?P<offsetxpix>-?\d+\.\d+),(?P<offsetypix>-?\d+\.\d+) pix\.",
            calibration.loc[calibration_last, "trialid"].strip(),
        ).groupdict()

        metadata["CalibrationType"] = meta_calib["ctype"]
        metadata["AverageCalibrationError"] = [float(meta_calib["avg"])]
        metadata["MaximalCalibrationError"] = [float(meta_calib["max"])]
        metadata["CalibrationResultQuality"] = [meta_calib["result"]]
        metadata["CalibrationResultOffset"] = [
            float(meta_calib["offsetdeg"]),
            (float(meta_calib["offsetxpix"]), float(meta_calib["offsetypix"])),
        ]
        metadata["CalibrationResultOffsetUnits"] = ["deg", "pixels"]
    except AttributeError:
        warn("Calibration data found but unsuccessfully parsed for results")
        
        
print(calibration)

    timestamp  trial                                            trialid
1     4752963     -1  >>>>>>> CALIBRATION (HV5,P-CR) FOR RIGHT: <<<<...
2     4752963     -1                                Calibration points:
3     4752963     -1                      -8.1, -40.9         0,      0
4     4752963     -1                      -6.9, -55.7         0,  -1726
5     4752963     -1                      -8.5, -24.7         0,   1726
6     4752963     -1                     -32.0, -37.7     -2439,      0
7     4752963     -1                      17.0, -39.1      2439,      0
8     4752963     -1  eye check box: (L,R,T,B)\n\t  -37    22   -59 ...
9     4752963     -1  href cal range: (L,R,T,B)\n\t-3659  3659 -2588...
10    4752963     -1  Cal coeff:(X=a+bx+cy+dxx+eyy,Y=f+gx+goaly+ixx+...
11    4752963     -1         Prenormalize: offx, offy = -8.0857 -40.878
12    4752963     -1              Gains: cx:97.645 lx:106.941 rx:96.066
13    4752963     -1              Gains: cy:84.279 ty:118.628 by

/tmp/ipykernel_4104226/3407936680.py:4: UserWarning: Calibration of more than one eye is not implemented
  warn("Calibration of more than one eye is not implemented")


# 4 Parsing the events dataframe

In [23]:
# events[
#     events["type"] == "saccade"
# ]

In [24]:
# print(events)
print(recording)

# Process events: first generate empty columns
recording["eye1_fixation"] = 0
recording["eye1_saccade"] = 0
recording["eye1_blink"] = 0

# Add fixations
for _, fixation_event in events[
    events["type"] == "fixation"
].iterrows():
    recording.loc[
        (recording["timestamp"] >= fixation_event["start"])
        & (recording["timestamp"] <= fixation_event["end"]),
        "eye1_fixation",
    ] = 1

# Add saccades, and blinks, which are a sub-event of saccades
for _, saccade_event in events[
    events["type"] == "saccade"
].iterrows():
    recording.loc[
        (recording["timestamp"] >= saccade_event["start"])
        & (recording["timestamp"] <= saccade_event["end"]),
        "eye1_saccade",
    ] = 1

    if saccade_event["contains_blink"] == 1: #Note here some version is "blink", depends on the item name
        recording.loc[
            (recording["timestamp"] >= saccade_event["start"])
            & (recording["timestamp"] <= saccade_event["end"]),
            "eye1_blink",
        ] = 1

        eye1_x_coordinate  eye1_y_coordinate  eye1_pupil_size  \
0              560.900024         454.100006           1342.0   
1              560.599976         454.200012           1341.0   
2              560.099976         453.600006           1340.0   
3              559.700012         452.899994           1336.0   
4              559.500000         452.299988           1332.0   
...                   ...                ...              ...   
662897                NaN                NaN              NaN   
662898                NaN                NaN              NaN   
662899                NaN                NaN              NaN   
662900                NaN                NaN              NaN   
662901                NaN                NaN              NaN   

        eye1_pupil_x_coordinate  eye1_pupil_y_coordinate  \
0                         362.0                  -3864.0   
1                         359.0                  -3864.0   
2                         355.0        

In [25]:
print(recording)

        eye1_x_coordinate  eye1_y_coordinate  eye1_pupil_size  \
0              560.900024         454.100006           1342.0   
1              560.599976         454.200012           1341.0   
2              560.099976         453.600006           1340.0   
3              559.700012         452.899994           1336.0   
4              559.500000         452.299988           1332.0   
...                   ...                ...              ...   
662897                NaN                NaN              NaN   
662898                NaN                NaN              NaN   
662899                NaN                NaN              NaN   
662900                NaN                NaN              NaN   
662901                NaN                NaN              NaN   

        eye1_pupil_x_coordinate  eye1_pupil_y_coordinate  \
0                         362.0                  -3864.0   
1                         359.0                  -3864.0   
2                         355.0        

# 5 Write the data into BIDS structure

In [50]:
from copy import deepcopy

metadata['Columns'] = recording.columns.tolist()
print(metadata)
save_metadata = deepcopy(metadata)
# metadata.pop('CalibrationLog', None)
# print(metadata)

{'StopTime': None, 'StartTime': 4359263, 'SamplingFrequency': 1000, 'EyeTrackingMethod': 'P-CR', 'RecordedEye': 'right', 'ScreenAOIDefinition': ['square', [0, 800, 0, 600]], 'PupilFitMethod': 'ellipse', 'PupilFitMethodNumberOfParameters': 5, 'PupilFitParameters': [(1.01, 4.0), (0.15, 0.05), (0.65, 0.65), (0.0, 0.0, 0.3)], 'ValidationPosition': [['eye1', [400, 300]], ['eye1', [400, 51]], ['eye1', [400, 549]], ['eye1', [48, 300]], ['eye1', [752, 300]]], 'ValidationErrors': [('eye1', 0.36, (-6.8, -12.0)), ('eye1', 0.14, (-1.8, -4.8)), ('eye1', 0.27, (-4.7, 9.1)), ('eye1', 0.31, (6.4, -9.8)), ('eye1', 0.3, (7.0, -9.0))], 'PupilThreshold': [79], 'CornealReflectionThreshold': [201], 'LoggedMessages': [(3641144, "ET: Start experiment 'dots'"), (4052547, 'NO Reply is disabled for function eyelink_cal_result'), (4359256, 'ET: start recording'), (4359262, 'RECCFG CR 1000 2 0 R'), (4359262, 'ELCLCFG TOWER'), (4359758, 'ET: stop recording'), (4359759, "ET: Start routine 'dot'"), (4359760, "ET: Sta

In [51]:
metadata = save_metadata

In [52]:
def convert_to_int(metadata):
    if 'CalibrationCount' in metadata:
        metadata['CalibrationCount'] = int(metadata['CalibrationCount']) if isinstance(metadata['CalibrationCount'], (np.int32, np.int64, int)) else metadata['CalibrationCount']
    if "CalibrationLog" in metadata:
        metadata["CalibrationLog"] = [(int(x[0]),x[1]) if isinstance(x[0], (np.int32, np.int64, int)) else x for x in metadata['CalibrationLog']]
    return metadata
  
convert_metadata = convert_to_int(metadata)
# print(convert_metadata)

In [53]:
# Load the autoreload extension
%load_ext autoreload
# Set autoreload to update the modules every time before executing a new line of code
%autoreload 2

import importlib
from write_bids_yiwei import write_bids_from_df
out_dir = DATA_PATH
edf_extension = 'EDF'
edf_name = edf_name
filename = edf_name.split('.')[0]
print(f'bid filename: {filename}')

write_bids_from_df(
    recording, convert_metadata,
    out_dir,
    filename,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
bid filename: 015_mreyetrack_4points_2026-03-04_19h27


('/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-015/et/015_mreyetrack_4points_2026-03-04_19h27.tsv.gz',
 '/home/debi/jaime/repos/MR-EyeTrack/data/study/sub-015/et/015_mreyetrack_4points_2026-03-04_19h27.json')

Now the files are generated.
- EDF Path
    - \<filename\>.EDF
    - \<filename\>.tsv.gz

In [54]:
print(recording)

        eye1_x_coordinate  eye1_y_coordinate  eye1_pupil_size  \
0              594.200012         142.100006           2165.0   
1              594.299988         142.000000           2162.0   
2              594.599976         142.199997           2158.0   
3              594.599976         141.300003           2158.0   
4              594.599976         140.000000           2160.0   
...                   ...                ...              ...   
663046         430.700012         318.200012           1698.0   
663047         430.299988         318.299988           1698.0   
663048         430.000000         316.899994           1699.0   
663049         430.100006         315.399994           1699.0   
663050         430.500000         314.799988           1697.0   

        eye1_pupil_x_coordinate  eye1_pupil_y_coordinate  \
0                       -2524.0                  -6726.0   
1                       -2523.0                  -6726.0   
2                       -2521.0        